In [ ]:
#  удаляем системный браузер, чтобы избежать конфликтов
!apt-get remove -y chromium-browser chromium-chromedriver
!apt-get autoremove -y

!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get update
!apt-get install -y google-chrome-stable

!pip install -U selenium beautifulsoup4 pandas

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import random

class KHLSeasonScraper:
    def __init__(self):
        self.driver = None
        self.setup_driver()

    def setup_driver(self):
        """перезапуск браузера, которая нужна чтобы версии браузера и силениума соответствовали друг другу и не было утечек памяти"""
        if self.driver:
            self.driver.quit()

        options = Options()
        options.add_argument('--headless=new')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)

        self.driver = webdriver.Chrome(options=options)
        self.driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
            'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
        })

    def parse_single_match(self, match_id):
        """парсинг одного матча"""
        url = f"https://text.khl.ru/text/{match_id}.html"
        self.driver.get(url)

        try:
            # Ждем прогрузки хотя бы одной точки (максимум 5 сек)
            WebDriverWait(self.driver, 5).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "circle.rink-circle-4"))
            )
        except:
            return []

        soup = BeautifulSoup(self.driver.page_source, 'html.parser')
        shots_data = []

        # убираем дубликаты
        full_match_map = soup.find('svg', id='shot-map-0')
        if not full_match_map:
            return []

        circles = full_match_map.find_all('circle', class_='rink-circle-4')

        for c in circles:
            try:
                x_coord = c.get('cx')
                y_coord = c.get('cy')
                shot_type = c.get('data-shot-type')

                # id игрока
                player_id = c.get('data-player')

                title_soup = BeautifulSoup(c.get('data-title', ''), 'html.parser')
                divs = title_soup.find_all('div')

                if len(divs) >= 3:
                    team_info = divs[0].text.replace('Гол. ', '').strip()

                    # выбираем номер, имя и амплуа игрока, например Пример: "58. Миллер Митчелл (з)" (топ1 защитник всей лиги сейчас кстати и мой любимый игрок:)
                    raw_player = divs[1].text

                    # разбиваем номер, имя и амплуа игрока
                    if '.' in raw_player:
                        player_number = raw_player.split('.')[0].strip()
                        name_and_pos = raw_player.split('.', 1)[1].strip()

                        if '(' in name_and_pos:
                            player_name = name_and_pos.split('(')[0].strip()
                            player_position = name_and_pos.split('(')[1].replace(')', '').strip()
                        else:
                            player_name = name_and_pos
                            player_position = ""
                    else:
                        player_number = ""
                        player_name = raw_player
                        player_position = ""

                    shots_data.append({
                        'match_id': match_id,
                        'team': team_info,
                        'player_id': player_id,
                        'player_number': player_number,
                        'player_name': player_name,
                        'player_position': player_position,
                        'time': divs[2].text.strip(),
                        'x_coord': int(x_coord),
                        'y_coord': int(y_coord),
                        'is_goal': 1 if shot_type == '1' else 0
                    })
            except Exception:
                continue

        return shots_data

    def run_season(self, match_ids, output_file="all_khl_shots.csv"):
        """парсинг всех броско за сезон"""

        processed_ids = set()
        if os.path.exists(output_file): # на случай, если парсинг прервался
            df_existing = pd.read_csv(output_file)
            processed_ids = set(df_existing['match_id'].astype(str).unique())
            print(f"{len(processed_ids)} матчей уже обработано. To be continued ...")
        else:
            cols = ['match_id', 'team', 'player_id', 'player_number', 'player_name', 'player_position', 'time', 'x_coord', 'y_coord', 'is_goal']
            pd.DataFrame(columns=cols).to_csv(output_file, index=False)

        matches_to_process = [m for m in match_ids if str(m) not in processed_ids]
        print(f"осталось: {len(matches_to_process)} матчей")

        for i, match_id in enumerate(matches_to_process, 1):
            print(f"[{i}/{len(matches_to_process)}] парсинг матча {match_id}", end=" ")

            if i % 50 == 0:
                self.setup_driver()

            try:
                shots = self.parse_single_match(match_id)
                if shots:
                    df_shots = pd.DataFrame(shots)
                    df_shots = df_shots.drop_duplicates()

                    df_shots.to_csv(output_file, mode='a', header=False, index=False, encoding='utf-8-sig')
                    print(f"({len(df_shots)} бросков)")
                else:
                    print("Нет карты бросков.(ну вдруг)")
            except Exception as e:
                print(f"Ошибка: {e}")

            # пауза чтоб нас не спалили
            time.sleep(random.uniform(1.5, 3.0))

    def close(self):
        if self.driver:
            self.driver.quit()

if __name__ == "__main__":
    try:
        df_calendar = pd.read_csv("khl_calendar.csv")
        df_calendar = df_calendar.dropna(subset=['link'])
        df_calendar['match_id'] = df_calendar['link'].str.extract(r'/game/\d+/(\d+)/')
        all_match_ids = df_calendar['match_id'].dropna().tolist()

        scraper = KHLSeasonScraper()
        scraper.run_season(all_match_ids)

    except FileNotFoundError:
        print("помести файл khl_calendar.csv в директорию этого кода!")
    finally:
        if 'scraper' in locals():
            scraper.close()
            print("\nВ-С-Ё")